# RSNA Knee Abnormality Detection — Image Baseline Preflight Audit

Before freezing an image-based baseline pipeline, this notebook measures several properties of the real competition DICOM data that pipeline choices depend on: whether `Fluid_Sensitive` and `Fat_Suppression` carry independent information, how many studies have usable coverage across the three anatomical planes, whether slice geometry tags are present and agree with `InstanceNumber` ordering, whether the `Laterality` tag is reliable, DICOM decode reliability, and a GPU timing probe for a frozen pretrained image encoder against the competition's runtime budget. Every result below is an aggregate count, rate, or distribution statistic — no report text, no study identifiers, and no per-study predictions.

In [ ]:
import json
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import pydicom
from IPython.display import display

SEED = 42
IS_KAGGLE = Path("/kaggle/input").exists()
if not IS_KAGGLE:
    raise RuntimeError("This notebook runs on Kaggle only.")

DATA_DIR = Path("/kaggle/input/competitions/rsna-knee-abnormality-detection")
package_initializers = tuple(
    Path("/kaggle/input/datasets").rglob("knee_mri/__init__.py")
)
if len(package_initializers) != 1:
    raise RuntimeError("Expected exactly one attached knee_mri source package.")
sys.path.insert(0, str(package_initializers[0].parent.parent))

In [ ]:
from knee_mri.series_audit import (
    audit_series,
    central_band_indices,
    fluid_fat_suppression_agreement,
    plane_series_counts,
)

## 1. Series Metadata Agreement and Plane Coverage (Full Corpus)

In [ ]:
train_series_df = pd.read_csv(DATA_DIR / "train_series.csv")
test_series_df = pd.read_csv(DATA_DIR / "test_series.csv")

agreement_summary = pd.DataFrame(
    [
        {"Split": "train", **fluid_fat_suppression_agreement(train_series_df)},
        {"Split": "test", **fluid_fat_suppression_agreement(test_series_df)},
    ]
).set_index("Split")

display(agreement_summary)

**Interpretation:** `agreement_rate` of 1.0 means `Fluid_Sensitive` and `Fat_Suppression` take the identical value on every series in that split — the two columns would carry no independent information, and any pipeline step that treats them as separate signals should instead treat them as one. A rate below 1.0 means the columns do sometimes disagree and remain two distinct signals worth keeping separate.

In [ ]:
def _plane_coverage(series_df: pd.DataFrame) -> pd.Series:
    plane_counts = plane_series_counts(series_df)
    present = plane_counts > 0
    coverage = {
        f"has_{plane.lower()}": float(present[plane].mean()) for plane in present.columns
    }
    coverage["has_all_three_planes"] = float(present.all(axis=1).mean())
    coverage["study_count"] = float(len(plane_counts))
    return pd.Series(coverage)

plane_coverage_summary = pd.DataFrame(
    {"train": _plane_coverage(train_series_df), "test": _plane_coverage(test_series_df)}
)

display(plane_coverage_summary)

**Interpretation:** `has_<plane>` is the fraction of studies with at least one series in that plane; `has_all_three_planes` is the fraction with usable coverage across Sagittal, Coronal, and Axial simultaneously. A high `has_all_three_planes` rate is what would make a multi-plane image baseline feasible without a large fallback-handling burden; a low rate favors a single-series baseline or an explicit per-study presence mask.

## 2. Deterministic DICOM Geometry and Decode Audit (Sampled Studies)

Auditing every series in the full corpus would decode a large fraction of the competition's 569.76 GB of DICOM data. Instead, this section samples a fixed, seeded set of studies and audits every series belonging to them — large enough for stable aggregate rates, small enough to run well inside the competition's runtime budget.

In [ ]:
import importlib.util

CODEC_PACKAGES = ("pylibjpeg", "pylibjpeg_libjpeg", "pylibjpeg_openjpeg", "gdcm")
codec_availability = pd.Series(
    {package: importlib.util.find_spec(package) is not None for package in CODEC_PACKAGES},
    name="Available",
).to_frame()

display(codec_availability)

**Interpretation:** the competition data mixes JPEG Lossless, JPEG 2000, and uncompressed transfer syntaxes; decoding the compressed ones needs one of these codec packages available. If none are available, the `Decode failure rate` below would reflect a missing-codec environment gap rather than genuinely corrupt DICOM data, and the pipeline would need to vendor a codec package offline before that number reflects the data rather than the environment.

In [ ]:
STUDY_SAMPLE_SIZE = 150
DECODE_SAMPLE_SIZE = 5

rng = np.random.default_rng(SEED)
all_study_ids = train_series_df["StudyInstanceUID"].unique()
sampled_study_ids = rng.choice(
    all_study_ids, size=min(STUDY_SAMPLE_SIZE, len(all_study_ids)), replace=False
)

train_series_dir = DATA_DIR / "train_series"
audit_rows = []
sampled_series_dirs = []
for study_id in sampled_study_ids:
    study_dir = train_series_dir / study_id
    if not study_dir.is_dir():
        continue
    for series_dir in sorted(p for p in study_dir.iterdir() if p.is_dir()):
        result = audit_series(series_dir, decode_sample_size=DECODE_SAMPLE_SIZE)
        audit_rows.append(
            {
                "slice_count": result.slice_count,
                "has_full_geometry_tags": result.has_full_geometry_tags,
                "order_agreement": result.order_agreement,
                "has_laterality_tag": result.has_laterality_tag,
                "laterality_from_geometry": result.laterality_from_geometry,
                "laterality_conflict": result.laterality_conflict,
                "pixel_spacing_row_mm": (
                    result.pixel_spacing[0] if result.pixel_spacing else None
                ),
                "pixel_spacing_col_mm": (
                    result.pixel_spacing[1] if result.pixel_spacing else None
                ),
                "decode_attempted": result.decode_attempted,
                "decode_failures": result.decode_failures,
            }
        )
        sampled_series_dirs.append(series_dir)

audit_df = pd.DataFrame(audit_rows)

In [ ]:
resolved_agreement = audit_df["order_agreement"].dropna()
resolvable_laterality = audit_df.loc[
    audit_df["laterality_from_geometry"].notna() & audit_df["has_laterality_tag"],
    "laterality_conflict",
]

geometry_summary = pd.Series(
    {
        "Series audited": len(audit_df),
        "Studies sampled": len(sampled_study_ids),
        "Geometry tag coverage": float(audit_df["has_full_geometry_tags"].mean()),
        "Order agreement -- mean |r|": float(resolved_agreement.abs().mean()),
        "Order agreement -- fraction |r| > 0.99": float(
            (resolved_agreement.abs() > 0.99).mean()
        ),
        "Order agreement -- fraction |r| <= 0.9": float(
            (resolved_agreement.abs() <= 0.9).mean()
        ),
        "Laterality tag coverage": float(audit_df["has_laterality_tag"].mean()),
        "Laterality conflict rate (resolvable)": (
            float(resolvable_laterality.mean()) if len(resolvable_laterality) else float("nan")
        ),
        "Decode failure rate": float(
            audit_df["decode_failures"].sum() / audit_df["decode_attempted"].sum()
        ),
    },
    name="Value",
).to_frame()

display(geometry_summary)

**Interpretation:** `Order agreement -- mean |r|` close to 1.0 means `InstanceNumber` order and true DICOM-geometry order agree (a consistent reversal still counts as agreement, since `|r|` is used); values much below 1.0 for a meaningful fraction of series would mean `InstanceNumber` alone is not a reliable substitute for geometry-based ordering. `Laterality tag coverage` below 1.0 with a non-trivial `Laterality conflict rate` among resolvable series means the geometry-derived laterality call is needed, not optional. `Decode failure rate` above 0 means the pipeline needs an explicit fallback for unreadable slices rather than assuming every attempted decode succeeds.

In [ ]:
pixel_spacing_summary = audit_df[
    ["pixel_spacing_row_mm", "pixel_spacing_col_mm"]
].describe().T[["mean", "std", "min", "max"]]
slice_count_summary = (
    audit_df["slice_count"]
    .describe()[["mean", "std", "min", "50%", "max"]]
    .rename({"50%": "median"})
    .to_frame(name="Value")
)

display(pixel_spacing_summary)
display(slice_count_summary)

**Interpretation:** a wide `pixel_spacing` range confirms physical-extent (millimeter-based) cropping is necessary rather than a fixed-pixel crop, since a fixed pixel window would cover a different real-world area per study. The `slice_count` distribution informs how large a central-band slice sample can be without exceeding what most series actually contain.

## 3. Frozen DINOv2-Small Load and GPU Timing Probe

Loads the frozen, offline-vendored `facebook/dinov2-small` model attached as a Kaggle Model source and times both DICOM decode/preprocessing and the GPU forward pass on a sample of the same sampled series, to project total wall-clock time against the competition's runtime budget before any pipeline design is frozen.

In [ ]:
import importlib.metadata

import torch
from transformers import AutoModel

if not torch.cuda.is_available():
    raise RuntimeError("Expected a GPU-enabled kernel for the DINOv2 timing probe.")


def _find_dinov2_dir(root: Path) -> Path:
    for config_path in root.rglob("config.json"):
        try:
            config = json.loads(config_path.read_text())
        except (OSError, json.JSONDecodeError):
            continue
        if config.get("model_type") == "dinov2":
            return config_path.parent
    raise RuntimeError("Could not find an attached DINOv2 model source.")


DEVICE = torch.device("cuda")
dinov2_dir = _find_dinov2_dir(Path("/kaggle/input"))
dinov2 = AutoModel.from_pretrained(str(dinov2_dir), local_files_only=True).to(DEVICE).eval()
for parameter in dinov2.parameters():
    parameter.requires_grad_(False)

cuda_major, cuda_minor = torch.cuda.get_device_capability(0)
GPU_COMPATIBLE = f"sm_{cuda_major}{cuda_minor}" in torch.cuda.get_arch_list()

environment_summary = pd.Series(
    {
        "torch version": importlib.metadata.version("torch"),
        "transformers version": importlib.metadata.version("transformers"),
        "CUDA device": torch.cuda.get_device_name(0),
        "CUDA compute capability": f"{cuda_major}.{cuda_minor}",
        "GPU compatible with installed PyTorch build": GPU_COMPATIBLE,
        "DINOv2 parameters": sum(p.numel() for p in dinov2.parameters()),
    },
    name="Value",
).to_frame()

display(environment_summary)

**Interpretation:** confirms the offline `model_sources` vendoring pattern works under `enable_internet: false` and records the exact runtime environment the timing numbers below were measured on, since GPU timing is only meaningful alongside the hardware/library versions it was measured with. Kaggle's shared GPU pool can allocate an older accelerator (e.g. a P100) whose compute capability the preinstalled PyTorch build no longer supports -- `GPU compatible with installed PyTorch build` makes that visible instead of the run crashing partway through.

In [ ]:
IMAGE_SIZE = 336
GPU_TIMING_SERIES_SAMPLE = 30
GPU_BATCH_SIZE = 32


def _percentile_normalize(pixels: np.ndarray) -> np.ndarray:
    low, high = np.percentile(pixels, [1, 99])
    if high <= low:
        return np.zeros_like(pixels, dtype=np.float32)
    clipped = np.clip(pixels, low, high)
    return ((clipped - low) / (high - low)).astype(np.float32)


def _to_model_input(pixels: np.ndarray) -> torch.Tensor:
    tensor = torch.from_numpy(pixels).unsqueeze(0).unsqueeze(0)
    resized = torch.nn.functional.interpolate(
        tensor, size=(IMAGE_SIZE, IMAGE_SIZE), mode="bilinear", align_corners=False
    )
    return resized.repeat(1, 3, 1, 1).squeeze(0)


timing_series_dirs = sampled_series_dirs[:GPU_TIMING_SERIES_SAMPLE]

decode_seconds = None
gpu_seconds = None
batch_tensors = []
if GPU_COMPATIBLE:
    decode_start = time.perf_counter()
    for series_dir in timing_series_dirs:
        dcm_paths = sorted(series_dir.glob("*.dcm"))
        for index in central_band_indices(len(dcm_paths), DECODE_SAMPLE_SIZE):
            dataset = pydicom.dcmread(dcm_paths[index])
            pixels = _percentile_normalize(dataset.pixel_array.astype(np.float32))
            batch_tensors.append(_to_model_input(pixels))
    decode_seconds = time.perf_counter() - decode_start

    gpu_seconds = 0.0
    for start in range(0, len(batch_tensors), GPU_BATCH_SIZE):
        chunk = torch.stack(batch_tensors[start : start + GPU_BATCH_SIZE]).to(DEVICE)
        torch.cuda.synchronize()
        gpu_start = time.perf_counter()
        with torch.no_grad():
            _ = dinov2(pixel_values=chunk, interpolate_pos_encoding=True)
        torch.cuda.synchronize()
        gpu_seconds += time.perf_counter() - gpu_start

In [ ]:
if GPU_COMPATIBLE:
    slices_processed = len(batch_tensors)
    series_processed = len(timing_series_dirs)
    seconds_per_series = (decode_seconds + gpu_seconds) / series_processed
    avg_series_per_study = len(audit_df) / len(sampled_study_ids)
    train_study_total = int(train_series_df["StudyInstanceUID"].nunique())

    timing_summary = pd.Series(
        {
            "GPU timing measured": True,
            "Slices processed (timing sample)": slices_processed,
            "Series processed (timing sample)": series_processed,
            "Decode seconds per slice": decode_seconds / slices_processed,
            "GPU forward seconds per slice": gpu_seconds / slices_processed,
            "Seconds per series (decode + GPU)": seconds_per_series,
            "Avg series per sampled study": avg_series_per_study,
            "Projected hours -- one series per study, full train": (
                seconds_per_series * train_study_total / 3600
            ),
            "Projected hours -- all series per study, full train": (
                seconds_per_series * avg_series_per_study * train_study_total / 3600
            ),
            "Competition runtime budget (hours)": 9.0,
        },
        name="Value",
    ).to_frame()
else:
    timing_summary = pd.Series(
        {
            "GPU timing measured": False,
            "Reason": "Allocated GPU compute capability unsupported by installed PyTorch",
        },
        name="Value",
    ).to_frame()

display(timing_summary)

**Interpretation:** the two projected-hours rows bound the wall-clock cost of encoding the full train split under a single-series-per-study design versus an all-series (multi-plane) design; comparing both against the competition's runtime budget is a direct, measured input into the single-series-vs-multi-plane scope decision, rather than an estimate inherited from a different notebook's hardware. If `GPU compatible with installed PyTorch build` was `False` above, `GPU timing measured` is `False` here and no projection could be computed this run -- re-running the kernel is the next step, since Kaggle's GPU allocation varies session to session.

## 4. Summary

This audit answers, with real measurements on this competition's data, the open questions raised before freezing an image baseline pipeline: whether `Fluid_Sensitive` and `Fat_Suppression` are redundant, how much multi-plane coverage exists per study, whether `InstanceNumber` order is a reliable substitute for geometry-based ordering, how reliable the `Laterality` tag is, DICOM decode reliability, and a measured GPU runtime projection against the competition's budget. These numbers, not assumptions carried over from public reference notebooks, are what the frozen pipeline design should be built on.